# Phase 3 — Audit : mesurer la qualité du dataset
**Durée estimée : 1h15**

---

## Objectif
Un dataset collecté n'est **pas automatiquement exploitable** pour un projet IA.  
Dans cette phase, vous allez mesurer sa qualité sur **6 dimensions** reconnues en ingénierie des données,  
puis produire un verdict chiffré (`/10`) à destination du rapport de synthèse.

## Les 6 dimensions de la qualité
| # | Dimension | Question clé |
|---|-----------|-------------|
| 1 | **Complétude** | Y a-t-il des valeurs manquantes ? |
| 2 | **Validité** | Les valeurs respectent-elles un format attendu ? |
| 3 | **Unicité** | Y a-t-il des doublons ? |
| 4 | **Exactitude** | Les dates sont-elles au bon format ISO ? |
| 5 | **Cohérence** | Les valeurs sont-elles logiquement cohérentes ? |
| 6 | **Actualité** | Les données sont-elles récentes (< 12 mois) ? |

## Livrables de cette phase
- Un rapport de profilage HTML (`rapport_audit.html`)
- Une cellule de résultats par dimension (OK / KO + pourquoi)
- Une note globale `/10`

---
> **Dataset de secours :** si vous n'avez pas de `data/export.csv` valide,  
> passez `UTILISER_SECOURS = True` dans la cellule suivante.

In [6]:
# --- Installation (décommentez si vous êtes sur Google Colab) ---
# !pip install pandas ydata-profiling great-expectations lxml ipywidgets

In [7]:
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime, timedelta
from IPython.display import display, Markdown, HTML

warnings.filterwarnings('ignore')

print("Imports OK ✓")

Imports OK ✓


---
## Étape 1 — Chargement du dataset

In [8]:
# ✏️  Passez à True si vous utilisez le dataset de secours
UTILISER_SECOURS = False

if UTILISER_SECOURS:
    chemin = "data/dataset_secours.csv"
    print("ℹ️  Mode secours activé — utilisation de data/dataset_secours.csv")
else:
    chemin = "data/export.csv"

if not os.path.exists(chemin):
    print(f"⚠️  Fichier '{chemin}' introuvable.")
    print("   → Passez UTILISER_SECOURS = True pour utiliser le dataset de secours.")
else:
    df = pd.read_csv(chemin, dtype=str)  # tout en string pour l'audit — on ne présuppose pas les types
    print(f"✅ Dataset chargé : {len(df)} lignes, {len(df.columns)} colonnes")
    print(f"   Colonnes : {list(df.columns)}")
    print()
    display(df.head(5))

✅ Dataset chargé : 30 lignes, 4 colonnes
   Colonnes : ['source', 'titre', 'date', 'url']



,source,titre,date,url
0,data.gouv.fr,Education,2019-03-19,https://www.data.gouv.fr/fr/datasets/education-1/
1,data.gouv.fr,Académies Education Prioritaire,2026-01-29,https://www.data.gouv.fr/fr/datasets/academies...
2,data.gouv.fr,OpenStreetMap : Education - points,2025-06-07,https://www.data.gouv.fr/fr/datasets/openstree...
3,data.gouv.fr,Séries chronologiques Education,2018-06-12,https://www.data.gouv.fr/fr/datasets/series-ch...
4,data.gouv.fr,OpenStreetMap : Education - polygones,2025-06-04,https://www.data.gouv.fr/fr/datasets/openstree...


---
## Étape 2 — Vérification du schéma minimal

Avant tout audit, on s'assure que le DataFrame a bien les 4 colonnes attendues.

In [9]:
COLONNES_ATTENDUES = ["source", "titre", "date", "url"]
colonnes_manquantes = [c for c in COLONNES_ATTENDUES if c not in df.columns]

if colonnes_manquantes:
    print(f"❌ Colonnes manquantes : {colonnes_manquantes}")
    print("   Ajoutez-les avant de continuer (valeur vide acceptée).")
else:
    print("✅ Schéma OK — les 4 colonnes sont présentes")
    print()
    print("=== Aperçu des types et valeurs manquantes ===")
    print(df.info())
    print()
    print("Valeurs manquantes par colonne :")
    print(df.isna().sum())

✅ Schéma OK — les 4 colonnes sont présentes

=== Aperçu des types et valeurs manquantes ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   source  30 non-null     object
 1   titre   30 non-null     object
 2   date    30 non-null     object
 3   url     30 non-null     object
dtypes: object(4)
memory usage: 1.1+ KB
None

Valeurs manquantes par colonne :
source    0
titre     0
date      0
url       0
dtype: int64


---
## Étape 3 — Profilage automatique (ydata-profiling)

Cette cellule génère un rapport HTML interactif.  
**Ouvrez `rapport_audit.html`** dans votre navigateur pour explorer les distributions, corrélations et alertes.

In [10]:
try:
    from ydata_profiling import ProfileReport

    profile = ProfileReport(
        df,
        title="Audit Qualité IA — Dataset Scraping",
        explorative=True,
        minimal=False,
    )
    profile.to_file("rapport_audit.html")
    print("✅ Rapport généré : rapport_audit.html")
    print("   → Ouvrez ce fichier dans votre navigateur pour l'explorer.")

except ImportError:
    print("⚠️  ydata-profiling non disponible — passage au profilage manuel.")
    print()
    print("=== df.info() ===")
    df.info()
    print()
    print("=== df.describe(include='all') ===")
    display(df.describe(include='all'))
    print()
    print("=== Valeurs manquantes ===")
    display(df.isna().sum().rename("manquantes"))

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 67.44it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Rapport généré : rapport_audit.html
   → Ouvrez ce fichier dans votre navigateur pour l'explorer.


---
## Étape 4 — Analyse du rapport de profilage

Après avoir ouvert `rapport_audit.html`, répondez aux questions ci-dessous.

### Vos observations du rapport (à compléter)

**Variable avec le plus de valeurs manquantes :**  
*à compléter — ex : `date` : 12 % de valeurs vides*

**Alerte / anomalie la plus importante signalée par ydata-profiling :**  
*à compléter — ex : 8 doublons détectés sur la colonne `url`*

**Observation sur la distribution de `source` :**  
*à compléter — ex : 100 % des valeurs sont "wikipedia.fr" → dataset mono-source*

**Première action de nettoyage que vous feriez :**  
*à compléter*

---
## Étape 5 — Tests des 6 dimensions

### Parcours A (pandas) — recommandé

Pour chaque dimension : le code est fourni, **exécutez-le et lisez le résultat**,  
puis complétez la cellule de conclusion juste en dessous.

---
### Dimension 1 — Complétude
> La colonne `titre` ne doit avoir **aucune valeur nulle**.

In [ ]:
# --- Dimension 1 : Complétude ---
titres_vides = df["titre"].isna() | (df["titre"].str.strip() == "")
nb_vides = titres_vides.sum()
taux_completude = (1 - nb_vides / len(df)) * 100

print(f"Titres vides ou nuls   : {nb_vides} / {len(df)}")
print(f"Taux de complétude     : {taux_completude:.1f} %")

if nb_vides > 0:
    print("\nLignes concernées :")
    display(df[titres_vides][["source", "titre", "url"]])

resultat_1 = "✅ OK" if nb_vides == 0 else f"❌ KO — {nb_vides} titre(s) manquant(s)"
print(f"\nRésultat D1 : {resultat_1}")

**Conclusion Dimension 1 — Complétude :**  
*à compléter — ex : OK, tous les titres sont renseignés / KO : 2 titres vides, lignes 8 et 13*

---
### Dimension 2 — Validité
> Les `url` doivent commencer par `http`.

In [ ]:
# --- Dimension 2 : Validité ---
urls_invalides = df["url"].isna() | ~df["url"].str.startswith("http")
nb_invalides = urls_invalides.sum()

print(f"URLs valides (http/https) : {len(df) - nb_invalides} / {len(df)}")
print(f"URLs invalides            : {nb_invalides}")

if nb_invalides > 0:
    print("\nURLs problématiques :")
    display(df[urls_invalides][["titre", "url"]])

resultat_2 = "✅ OK" if nb_invalides == 0 else f"❌ KO — {nb_invalides} URL(s) invalide(s)"
print(f"\nRésultat D2 : {resultat_2}")

**Conclusion Dimension 2 — Validité :**  
*à compléter*

---
### Dimension 3 — Unicité
> Absence de doublons sur le couple `titre` / `url`.

In [ ]:
# --- Dimension 3 : Unicité ---
masque_doublons = df.duplicated(subset=["titre", "url"], keep=False)
nb_doublons_lignes = masque_doublons.sum()
nb_groupes = df[masque_doublons].groupby(["titre", "url"]).ngroups if nb_doublons_lignes > 0 else 0

print(f"Lignes en doublon (titre+url) : {nb_doublons_lignes}")
print(f"Groupes de doublons distincts : {nb_groupes}")

if nb_doublons_lignes > 0:
    print("\nExemples de doublons :")
    display(df[masque_doublons].sort_values(["titre", "url"]).head(10))

resultat_3 = "✅ OK" if nb_doublons_lignes == 0 else f"❌ KO — {nb_doublons_lignes} lignes en doublon ({nb_groupes} groupes)"
print(f"\nRésultat D3 : {resultat_3}")

**Conclusion Dimension 3 — Unicité :**  
*à compléter*

---
### Dimension 4 — Exactitude
> Les dates doivent être au **format ISO** `AAAA-MM-JJ`.

In [ ]:
# --- Dimension 4 : Exactitude ---
import re

df_avec_date = df[df["date"].notna() & (df["date"].str.strip() != "")].copy()
df_sans_date = df[df["date"].isna() | (df["date"].str.strip() == "")].copy()

print(f"Lignes avec une date  : {len(df_avec_date)}")
print(f"Lignes sans date      : {len(df_sans_date)}")

if len(df_avec_date) == 0:
    print("\n⚠️  La colonne 'date' est entièrement vide.")
    print("   Cas typique : pages de catégorie Wikipedia (pas de date par article).")
    print("   → Dimension NON ÉVALUABLE — documentez-le dans votre conclusion.")
    nb_format_ok = nb_format_ko = 0
    resultat_4 = "⚠️  NON ÉVALUABLE — source sans dates"
else:
    pattern_iso = r'^\d{4}-\d{2}-\d{2}$'
    df_avec_date["format_iso"] = df_avec_date["date"].str.match(pattern_iso)
    nb_format_ok = df_avec_date["format_iso"].sum()
    nb_format_ko = (~df_avec_date["format_iso"]).sum()

    print(f"\nParmi les dates renseignées :")
    print(f"  Format ISO correct  : {nb_format_ok}")
    print(f"  Format incorrect    : {nb_format_ko}")

    if nb_format_ko > 0:
        print("\nDates au mauvais format :")
        display(df_avec_date[~df_avec_date["format_iso"]][["titre", "date"]])

    resultat_4 = "✅ OK" if nb_format_ko == 0 else f"❌ KO — {nb_format_ko} date(s) hors format ISO"

print(f"\nRésultat D4 : {resultat_4}")

**Conclusion Dimension 4 — Exactitude :**  
*à compléter*

> **Note :** Si votre source ne fournit pas de dates (ex : pages de catégorie Wikipedia sans date),  
> documentez-le ici plutôt que de forcer des valeurs.

---
### Dimension 5 — Cohérence
> **Règle du TP :** si la source est un **Média**, le titre doit contenir plus de 5 caractères.  
> Pour les autres sources (Wikipedia, data.gouv.fr), on vérifie l'absence de titres "fantômes" (≤ 2 caractères).

**Pourquoi ≤ 5 pour les Médias ?** Un titre journalistique trop court signale souvent un champ mal rempli ou un artefact d'extraction. À l'inverse, des acronymes légitimes (ex: `Keras`, `Lisp`, `DABUS`) apparaissent fréquemment dans Wikipedia : ils ne sont pas des anomalies.

In [ ]:
# --- Dimension 5 : Cohérence ---

# Mots-clés qui identifient une source de type Média
SOURCES_MEDIA = ["lemonde", "liberation", "figaro", "media", "média", "presse", "journal", "actu"]
masque_media = df["source"].str.lower().str.contains("|".join(SOURCES_MEDIA), na=False)
lignes_media = df[masque_media]

if len(lignes_media) > 0:
    # Règle stricte pour les Médias : titre > 5 chars
    df_titres_courts = lignes_media[
        lignes_media["titre"].notna() & (lignes_media["titre"].str.strip().str.len() <= 5)
    ]
    scope_info = f"sources Média identifiées : {len(lignes_media)} lignes"
else:
    # Aucune source Média → règle plus souple : titres ≤ 2 chars (vrais "fantômes")
    # Cela exclut les acronymes légitimes type Keras (5), Lisp (4), DABUS (5)
    df_titres_courts = df[
        df["titre"].notna() & (df["titre"].str.strip().str.len() <= 2)
    ]
    scope_info = "aucune source Média → seuil élargi (≤ 2 chars, pour titres fantômes uniquement)"

nb_courts = len(df_titres_courts)
print(f"Scope : {scope_info}")
print(f"Titres suspects : {nb_courts}")

if nb_courts > 0:
    print("\nLignes signalées :")
    display(df_titres_courts[["source", "titre", "url"]])

# Statistiques de longueur (utiles pour le rapport)
longueurs = df["titre"].dropna().str.len()
print(f"\nLongueur des titres — min: {longueurs.min():.0f} | médiane: {longueurs.median():.0f} | moy: {longueurs.mean():.0f} | max: {longueurs.max():.0f}")

resultat_5 = "✅ OK" if nb_courts == 0 else f"⚠️  ATTENTION — {nb_courts} titre(s) suspect(s)"
print(f"\nRésultat D5 : {resultat_5}")

**Conclusion Dimension 5 — Cohérence :**  
*à compléter*

---
### Dimension 6 — Actualité
> Le dataset contient-il des données de **moins de 12 mois** ?

In [ ]:
# --- Dimension 6 : Actualité ---
aujourd_hui = datetime.today()
limite_12_mois = aujourd_hui - timedelta(days=365)

print(f"Date du jour         : {aujourd_hui.strftime('%Y-%m-%d')}")
print(f"Seuil 12 mois        : {limite_12_mois.strftime('%Y-%m-%d')}")

# On tente de parser les dates en format ISO
dates_parsees = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")

nb_parseables   = dates_parsees.notna().sum()
nb_recentes     = (dates_parsees >= limite_12_mois).sum()
nb_anciennes    = (dates_parsees < limite_12_mois).sum()
nb_non_parsees  = len(df) - nb_parseables

print(f"\nDates parsées        : {nb_parseables} / {len(df)}")
print(f"  Récentes (< 12 mois) : {nb_recentes}")
print(f"  Anciennes (> 12 mois): {nb_anciennes}")
print(f"  Non parsées          : {nb_non_parsees}")

if nb_parseables > 0:
    pct_recentes = (nb_recentes / nb_parseables) * 100
    print(f"\nTaux d'actualité     : {pct_recentes:.0f} % des dates parsées")
    print(f"Date la plus récente : {dates_parsees.max().strftime('%Y-%m-%d') if dates_parsees.notna().any() else 'N/A'}")
    print(f"Date la plus ancienne: {dates_parsees.min().strftime('%Y-%m-%d') if dates_parsees.notna().any() else 'N/A'}")

if nb_parseables == 0:
    resultat_6 = "⚠️  NON ÉVALUABLE — aucune date parseable (colonne vide ou format non-ISO)"
elif nb_recentes > 0:
    resultat_6 = f"✅ OK — {nb_recentes} article(s) récent(s) (< 12 mois)"
else:
    resultat_6 = "❌ KO — aucun article récent dans les 12 derniers mois"

print(f"\nRésultat D6 : {resultat_6}")

**Conclusion Dimension 6 — Actualité :**  
*à compléter*

---
## Étape 6 — Tableau de bord récapitulatif

Exécutez cette cellule après avoir complété tous les tests : elle génère votre tableau de bord qualité.

In [ ]:
# --- Tableau de bord automatique ---

# Reconstruit toutes les variables (safe pour re-run partiel)
import re

titres_vides    = df["titre"].isna() | (df["titre"].str.strip() == "")
urls_invalides  = df["url"].isna() | ~df["url"].str.startswith("http")
masque_doublons = df.duplicated(subset=["titre", "url"], keep=False)

# D4 — Exactitude
df_avec_date = df[df["date"].notna() & (df["date"].str.strip() != "")].copy()
if len(df_avec_date) > 0:
    df_avec_date["format_iso"] = df_avec_date["date"].str.match(r'^\d{4}-\d{2}-\d{2}$')
    nb_format_ko = (~df_avec_date["format_iso"]).sum()
    d4_resultat  = "✅ OK" if nb_format_ko == 0 else f"❌ KO ({nb_format_ko} hors format)"
    d4_score     = 10 if nb_format_ko == 0 else max(0, round(10 * (1 - nb_format_ko / len(df_avec_date))))
else:
    nb_format_ko = 0
    d4_resultat  = "⚠️  NON ÉVALUABLE (colonne vide)"
    d4_score     = 5  # score neutre : ni bon ni mauvais

# D5 — Cohérence (adapté à la source)
SOURCES_MEDIA = ["lemonde", "liberation", "figaro", "media", "média", "presse", "journal", "actu"]
masque_media = df["source"].str.lower().str.contains("|".join(SOURCES_MEDIA), na=False)
lignes_media = df[masque_media]
if len(lignes_media) > 0:
    nb_courts = (lignes_media["titre"].notna() & (lignes_media["titre"].str.strip().str.len() <= 5)).sum()
else:
    nb_courts = (df["titre"].notna() & (df["titre"].str.strip().str.len() <= 2)).sum()

# D6 — Actualité
dates_parsees = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
nb_recentes   = (dates_parsees >= (datetime.today() - timedelta(days=365))).sum()

# Tableau de bord
dimensions = [
    {
        "Dimension":   "1 — Complétude",
        "Test":        "titres sans valeur nulle",
        "Résultat":    "✅ OK" if titres_vides.sum() == 0 else f"❌ KO ({titres_vides.sum()} vide(s))",
        "Score (/10)": 10 if titres_vides.sum() == 0 else max(0, round(10 * (1 - titres_vides.sum()/len(df)))),
    },
    {
        "Dimension":   "2 — Validité",
        "Test":        "URLs commençant par http",
        "Résultat":    "✅ OK" if urls_invalides.sum() == 0 else f"❌ KO ({urls_invalides.sum()} invalide(s))",
        "Score (/10)": 10 if urls_invalides.sum() == 0 else max(0, round(10 * (1 - urls_invalides.sum()/len(df)))),
    },
    {
        "Dimension":   "3 — Unicité",
        "Test":        "absence de doublons titre+url",
        "Résultat":    "✅ OK" if masque_doublons.sum() == 0 else f"❌ KO ({masque_doublons.sum()} lignes)",
        "Score (/10)": 10 if masque_doublons.sum() == 0 else max(0, round(10 * (1 - masque_doublons.sum()/len(df)))),
    },
    {
        "Dimension":   "4 — Exactitude",
        "Test":        "format date ISO AAAA-MM-JJ",
        "Résultat":    d4_resultat,
        "Score (/10)": d4_score,
    },
    {
        "Dimension":   "5 — Cohérence",
        "Test":        "titres > 5 chars (Média) ou > 2 chars (autres)",
        "Résultat":    "✅ OK" if nb_courts == 0 else f"⚠️  {nb_courts} titre(s) suspect(s)",
        "Score (/10)": 10 if nb_courts == 0 else 7,
    },
    {
        "Dimension":   "6 — Actualité",
        "Test":        "données < 12 mois",
        "Résultat":    "✅ OK" if nb_recentes > 0 else ("⚠️  NON ÉVALUABLE" if dates_parsees.notna().sum() == 0 else "❌ KO"),
        "Score (/10)": 10 if nb_recentes > 0 else (5 if dates_parsees.notna().sum() == 0 else 0),
    },
]

df_dashboard = pd.DataFrame(dimensions)
note_globale = df_dashboard["Score (/10)"].mean()

print("=" * 60)
print("         TABLEAU DE BORD QUALITÉ")
print("=" * 60)
display(df_dashboard)
print()
print(f"  NOTE GLOBALE : {note_globale:.1f} / 10")
print()
if note_globale >= 8:
    verdict = "✅ GO — dataset exploitable pour un projet IA"
elif note_globale >= 5:
    verdict = "⚠️  GO SOUS CONDITIONS — nettoyage requis avant usage"
else:
    verdict = "❌ NO-GO — trop d'anomalies, retraitement profond nécessaire"
print(f"  VERDICT PRÉLIMINAIRE : {verdict}")
print("=" * 60)

---
## Étape 7 — (Bonus) Parcours B : Great Expectations

Ce parcours est **optionnel** — il vous donne un aperçu des outils industriels d'audit de données.

> Great Expectations est utilisé en production dans de nombreuses entreprises pour "tester" les pipelines de données,  
> de la même façon que pytest teste du code.

> **⚠️ Note de version :** le code ci-dessous utilise l'API "legacy" de GE (style 0.x / `gx.from_pandas`).  
> Avec GE ≥ 1.7 (requis pour pandas 2.2), cette API n'existe plus → la cellule affichera  
> un message d'erreur informatif. Traitez-le comme un cas d'école sur les ruptures d'API.  
> L'API moderne GE 1.x utilise un `DataContext` — hors périmètre de ce TP.

In [ ]:
# ✏️  Passez à True pour activer le parcours Great Expectations
ACTIVER_GE = False

if ACTIVER_GE:
    try:
        import great_expectations as gx
        from great_expectations.dataset import PandasDataset

        ge_df = gx.from_pandas(df)

        resultats_ge = []

        # D1 — Complétude
        r = ge_df.expect_column_values_to_not_be_null("titre")
        resultats_ge.append({"Dimension": "1 — Complétude",  "Expectation": "titre non nul",         "Succès": r.success})

        # D2 — Validité
        r = ge_df.expect_column_values_to_match_regex("url", r"^https?://")
        resultats_ge.append({"Dimension": "2 — Validité",    "Expectation": "url commence par http", "Succès": r.success})

        # D3 — Unicité
        r = ge_df.expect_compound_columns_to_be_unique(["titre", "url"])
        resultats_ge.append({"Dimension": "3 — Unicité",     "Expectation": "(titre, url) unique",   "Succès": r.success})

        # D4 — Exactitude
        r = ge_df.expect_column_values_to_match_strftime_format("date", "%Y-%m-%d", mostly=0.9)
        resultats_ge.append({"Dimension": "4 — Exactitude",  "Expectation": "date ISO (90%+)",       "Succès": r.success})

        # D5 — Cohérence
        r = ge_df.expect_column_value_lengths_to_be_between("titre", min_value=6)
        resultats_ge.append({"Dimension": "5 — Cohérence",   "Expectation": "len(titre) > 5",        "Succès": r.success})

        display(pd.DataFrame(resultats_ge))

    except ImportError:
        print("⚠️  great-expectations non disponible. Installez-le avec : pip install great-expectations")
    except Exception as e:
        print(f"Erreur Great Expectations : {e}")
        print("Note : la syntaxe peut varier selon la version installée.")
else:
    print("Parcours B désactivé (ACTIVER_GE = False)")

---
## ✅ Fin de la Phase 3

Avant de passer au notebook `04_synthese.ipynb`, vérifiez :
- [ ] Vous avez exécuté les 6 tests et complété les cellules de conclusion
- [ ] Votre tableau de bord affiche une note globale
- [ ] `rapport_audit.html` est généré (ou profilage manuel fait)

**Notez votre note globale `/10` et 2 constats clés — vous en aurez besoin en Phase 4.**

**Passez maintenant à `04_synthese.ipynb` →**